# ⚡ RAPIDEX IA — Teste de Pipeline
**Antes de rodar:** Vá em `Ambiente de execução → Alterar tipo → T4 GPU`

Clique em `Ambiente de execução → Executar tudo` e aguarde.

In [ ]:
# Célula 1: Verificar GPU
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode==0 else '⚠️ CPU mode — mude para T4 GPU')
import sys; print('Python:', sys.version[:10])

In [ ]:
# Célula 2: Instalar dependências (~3 min)
!pip install -q gradio openai-whisper whisperx deep-translator demucs gtts TTS opencv-python matplotlib transformers accelerate ctranslate2
!pip install -q --upgrade click 'typer<0.25'
print('✅ Dependências instaladas')

In [ ]:
# Célula 3: Baixar código do GitHub
import os
if not os.path.exists('rapidex-ia'):
    !git clone -q https://github.com/tiagobignon6-blip/rapidex-ia.git
else:
    !cd rapidex-ia && git pull -q
os.environ['RAPIDEX_BASE'] = '/content'
os.environ['COQUI_TOS_AGREED'] = '1'
print('✅ Repo atualizado')
!ls rapidex-ia/

In [ ]:
# Célula 4: Criar vídeo de teste
from gtts import gTTS
import subprocess, os
gTTS('Hello, this is a test of the RAPIDEX IA dubbing pipeline. It should transcribe and translate this audio correctly.', lang='en').save('/tmp/test_audio.mp3')
!wget -q 'https://raw.githubusercontent.com/TMElyralab/MuseTalk/main/data/video/sun.mp4' -O /tmp/face.mp4 2>/dev/null
if not os.path.exists('/tmp/face.mp4') or os.path.getsize('/tmp/face.mp4') < 10000:
    subprocess.run(['ffmpeg','-y','-f','lavfi','-i','color=c=blue:size=320x240:rate=25','-t','12','/tmp/face.mp4'], capture_output=True)
subprocess.run(['ffmpeg','-y','-i','/tmp/face.mp4','-i','/tmp/test_audio.mp3','-map','0:v','-map','1:a','-c:v','copy','-shortest','/tmp/test.mp4'], capture_output=True)
print(f'✅ Vídeo de teste: {os.path.getsize("/tmp/test.mp4")/1024:.0f}KB')

In [ ]:
# Célula 5: Carregar funções do pipeline
import sys, os
os.chdir('/content')
src = open('rapidex-ia/app.py').read()
stop = src.find('\nCSS = ')
exec(src[:stop], globals())
print('✅ Pipeline carregado')

In [ ]:
# Célula 6-11: Testar cada etapa
import tempfile, time
tmp = tempfile.mkdtemp()
results = {}

print('[1/6] extract_audio...')
t=time.time(); raw = extract_audio('/tmp/test.mp4', tmp); results['extract']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s')

print('[2/6] run_demucs...')
t=time.time(); vocals, bg = run_demucs(raw, tmp); results['demucs']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s')

print('[3/6] run_whisperx (grande modelo, pode demorar ~2min)...')
t=time.time(); text = run_whisperx(vocals, 'en'); results['whisperx']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s | Texto: "{text[:80]}"')

print('[4/6] translate_text...')
t=time.time(); translated = translate_text(text, 'en', 'pt'); results['translate']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s | Tradução: "{translated[:80]}"')

print('[5/6] run_fish_speech (XTTS v2 ou gTTS fallback)...')
t=time.time(); dubbed = run_fish_speech(translated, vocals, tmp); results['tts']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s')

print('[6/6] run_lipsync (sem MuseTalk/Wav2Lip no Colab — substituição de áudio)...')
t=time.time(); out = run_lipsync('/tmp/test.mp4', mix_audio(dubbed, bg, tmp), tmp); results['lipsync']=(time.time()-t, True)
print(f'  ✅ OK em {time.time()-t:.1f}s')

print('\n=== RESUMO ===')
for k,(t,ok) in results.items():
    print(f'  {"✅" if ok else "❌"} {k}: {t:.1f}s')
print('\n🎉 Pipeline 100% funcional!')

In [ ]:
# Célula 7: Subir interface Gradio
import subprocess, os, time, threading
os.environ['RAPIDEX_BASE'] = '/content'
proc = subprocess.Popen(['python', 'rapidex-ia/app.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('Iniciando Gradio...')
for _ in range(60):
    line = proc.stdout.readline()
    if 'gradio.live' in line or 'Running on public' in line:
        print('🚀', line.strip())
        break
    elif line.strip():
        print(line.strip())